# Climate Feature Visualization (New Pipeline)

This notebook visualizes the new climate features created in:
- `data/ag/corn.csv`
- `data/ag/soybean.csv`
- `data/ag/wheat.csv`

It focuses on:
1. Time-series behavior of features
2. Feature distributions
3. Non-zero activity by feature and crop


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

DATA_DIR = Path('../data/ag')
CROPS = ['corn', 'soybean', 'wheat']

FEATURE_COLS = [
    'tmax_hot_in_planting', 'tmax_hot_in_harvesting',
    'tmax_very_hot_in_planting', 'tmax_very_hot_in_harvesting',
    'tmin_cold_in_planting', 'tmin_cold_in_harvesting',
    'tmin_very_cold_in_planting', 'tmin_very_cold_in_harvesting',
    'awnd_moderate_high_wind_in_planting', 'awnd_moderate_high_wind_in_harvesting',
    'awnd_extreme_high_wind_in_planting', 'awnd_extreme_high_wind_in_harvesting',
    'spi_7d_in_planting', 'spi_7d_in_harvesting',
    'spi_1m_in_planting', 'spi_1m_in_harvesting',
    'spi_3m_in_planting', 'spi_3m_in_harvesting',
    'pdsi_extreme_dry_in_planting', 'pdsi_extreme_dry_in_harvesting',
    'pdsi_dry_in_planting', 'pdsi_dry_in_harvesting',
    'pdsi_wet_in_planting', 'pdsi_wet_in_harvesting',
    'pdsi_extreme_wet_in_planting', 'pdsi_extreme_wet_in_harvesting',
    'co2_extreme_in_planting', 'co2_extreme_in_harvesting',
]


In [ ]:
dfs = {}
for crop in CROPS:
    path = DATA_DIR / f'{crop}.csv'
    df = pd.read_csv(path, parse_dates=['date'])
    missing = [c for c in FEATURE_COLS if c not in df.columns]
    if missing:
        print(f'[{crop}] Missing feature columns: {missing}')
    dfs[crop] = df
    print(f'[{crop}] rows={len(df):,}, cols={len(df.columns):,}')


## Helper Plot Functions


In [ ]:
def plot_feature_timeseries(feature, smooth_window=13):
    fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
    for idx, crop in enumerate(CROPS):
        df = dfs[crop][['date', feature]].copy()
        ax = axes[idx]
        ax.plot(df['date'], df[feature], alpha=0.35, linewidth=1, label='weekly')
        ax.plot(
            df['date'],
            df[feature].rolling(smooth_window, min_periods=1).mean(),
            linewidth=2,
            label=f'{smooth_window}-week MA',
        )
        ax.set_title(f'{crop.title()} - {feature}')
        ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()


def plot_feature_distribution(feature):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
    for idx, crop in enumerate(CROPS):
        s = dfs[crop][feature].dropna()
        sns.histplot(s, bins=40, kde=True, ax=axes[idx])
        axes[idx].set_title(f'{crop.title()} - {feature}')
    plt.tight_layout()
    plt.show()


def plot_feature_boxplot(feature):
    rows = []
    for crop in CROPS:
        tmp = dfs[crop][['date', feature]].copy()
        tmp['crop'] = crop
        rows.append(tmp)
    long_df = pd.concat(rows, ignore_index=True)

    plt.figure(figsize=(10, 5))
    sns.boxplot(data=long_df, x='crop', y=feature)
    plt.title(f'Cross-crop distribution: {feature}')
    plt.tight_layout()
    plt.show()


## Feature Activity Summary (Non-zero Rate)


In [ ]:
summary_rows = []
for crop in CROPS:
    df = dfs[crop]
    for feat in FEATURE_COLS:
        if feat not in df.columns:
            continue
        s = pd.to_numeric(df[feat], errors='coerce').fillna(0.0)
        summary_rows.append({
            'crop': crop,
            'feature': feat,
            'non_zero_rate': (s != 0).mean(),
            'mean': s.mean(),
            'std': s.std(),
            'q95': s.quantile(0.95),
        })

summary = pd.DataFrame(summary_rows).sort_values(['feature', 'crop']).reset_index(drop=True)
summary.head(15)


In [ ]:
pivot_nz = summary.pivot(index='feature', columns='crop', values='non_zero_rate')
plt.figure(figsize=(10, 12))
sns.heatmap(pivot_nz, cmap='YlOrRd', annot=True, fmt='.2f')
plt.title('Non-zero Rate by Feature and Crop')
plt.tight_layout()
plt.show()


## Plot Selected Features


In [ ]:
selected_features = [
    'tmax_very_hot_in_harvesting',
    'tmin_very_cold_in_planting',
    'awnd_extreme_high_wind_in_harvesting',
    'spi_3m_in_planting',
    'pdsi_extreme_dry_in_harvesting',
    'pdsi_extreme_wet_in_planting',
    'co2_extreme_in_harvesting',
]

for feature in selected_features:
    print(f'\n=== {feature} ===')
    plot_feature_timeseries(feature)
    plot_feature_distribution(feature)
    plot_feature_boxplot(feature)


## Optional: Plot All Features
Uncomment the loop below if you want full plotting for every feature (can be many plots).


In [ ]:
# for feature in FEATURE_COLS:
#     print(f'\n=== {feature} ===')
#     plot_feature_timeseries(feature)
#     plot_feature_distribution(feature)
#     plot_feature_boxplot(feature)
